In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

load_dotenv()

C:\Users\praja\AppData\Local\Temp\ipykernel_928\1786627429.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


True

In [2]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [6]:
text = """
            LangChain is a framework for building LLM applications.

            RAG stands for Retrieval Augmented Generation.

            Vector databases store embeddings and enable semantic search.

            Transformers are the foundation of modern Large Language Models.
            """

In [12]:
splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
documents=splitter.create_documents([text])

In [8]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
db=FAISS.from_documents(documents,embeddings)

In [14]:
retriever=db.as_retriever(search_kwargs={"k":2})

In [15]:
retriever.invoke("What is LangChain?")

[Document(id='5833f5c5-ddce-427e-9c9f-e56db144aae4', metadata={}, page_content='LangChain is a framework for building LLM applications.\n\n            RAG stands for Retrieval Augmented Generation.'),
 Document(id='3ee8c677-27ce-42b9-9448-6d0a69281ace', metadata={}, page_content='Vector databases store embeddings and enable semantic search.\n\n            Transformers are the foundation of modern Large Language Models.')]

##### Doc Chain

In [16]:
prompt=ChatPromptTemplate.from_template("""
Answer the question based on the provided context.

<context>
{context}
</context>

Question:
{input}
""")

In [18]:
document_chain=create_stuff_documents_chain(llm,prompt)

##### Retriever Chain

In [19]:
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [20]:
response=retrieval_chain.invoke({
    "input":"What is RAG?"
})

In [21]:
response.keys()

dict_keys(['input', 'context', 'answer'])

In [22]:
print(response["answer"])

RAG stands for Retrieval Augmented Generation.
